# 5. Model Evaluation and Final Labeling

This notebook evaluates the performance of the 100 trained models and determines the final, stable subtype assignment for each sample.

**Key Steps:**
1.  **Load Models:** Load the `state_dict` of all 100 models trained in the previous notebook.
2.  **Evaluate Performance:** Calculate the clustering accuracy (ACC) for each model against the K-means benchmark labels.
3.  **Majority Vote:** Generate cluster predictions from all models and assign the final subtype to each sample based on a majority vote.
4.  **Save Results:** Save the final clinical dataframe, which now includes the stable subtype labels, for the final analysis notebook.


### 5.1. Import Libraries and Configuration


In [ ]:
import sys
import torch
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from glob import glob
from scipy.stats import mode

# Add the project's 'src' directory to the Python path
sys.path.append('../src')

import config
from models import MIC
from utils import set_seed, cluster_accuracy, map_clusters_to_subtypes

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


### 5.2. Load Data and Models


In [ ]:
# Load the preprocessed data
data_path = config.PROCESSED_DATA_DIR / "processed_dataset.pt"
processed_data = torch.load(data_path, weights_only=False)
clinical_df = processed_data['clinical_df']

# Create a DataLoader for evaluation
eval_dataset = TensorDataset(
    processed_data['input_genotype'],
    processed_data['input_proteome'],
    processed_data['input_metabolite'],
    processed_data['output_clinical']
)
eval_loader = DataLoader(eval_dataset, batch_size=config.BATCH_SIZE)
print("Data loaded successfully.")

# Load all trained model state dicts
model_paths = sorted(glob(str(config.MODEL_SAVE_DIR / "*.pth")))
mic_models = [torch.load(path, map_location=device) for path in model_paths]
print(f"Successfully loaded {len(mic_models)} trained models.")


### 5.3. Evaluate Clustering Performance of All Models


In [ ]:
# Factorize the benchmark string labels into numeric labels for accuracy calculation
true_labels, _ = pd.factorize(clinical_df["kmeans_cluster"])

acc_list = []

input_dims = {
    'genotype': processed_data['input_genotype'].shape[1],
    'proteome': processed_data['input_proteome'].shape[1],
    'metabolite': processed_data['input_metabolite'].shape[1]
}

for i, state_dict in enumerate(mic_models):
    set_seed(100 + i)
    # Initialize a new model
    model = MIC(
        input_dims=input_dims,
        encoder_dims=config.ENCODER_DIMS,
        integration_dims=config.INTEGRATION_DIMS,
        latent_dim=config.LATENT_DIM,
        decoder_dims=config.DECODER_DIMS,
        clinical_output_dim=config.CLINICAL_OUTPUT_DIM,
        cluster_num=config.NUM_CLUSTERS,
        dropout=config.DROPOUT
    ).to(device)
    
    model.load_state_dict(state_dict)
    
    pred_labels = model.k_means_clustering(eval_loader, n_init=100, device=device)
    
    acc = cluster_accuracy(true_labels, pred_labels)
    acc_list.append(acc)

print("Evaluation complete.")


### 5.4. Determine Final Subtype Labels via Majority Vote


In [ ]:
# Factorize benchmark labels to ensure they are numeric for accuracy calculation
# Note: The cluster_accuracy function expects numeric labels.
true_labels_numeric, true_labels_str = pd.factorize(clinical_df["kmeans_cluster"])

valid_prediction_columns = []
all_mapped_predictions = pd.DataFrame(index=clinical_df.index)

print("Starting evaluation of 100 models...")
for i, state_dict in enumerate(mic_models):
    run_seed = 100 + i
    set_seed(run_seed)
    
    model = MIC(
        input_dims=input_dims,
        encoder_dims=config.ENCODER_DIMS,
        integration_dims=config.INTEGRATION_DIMS,
        latent_dim=config.LATENT_DIM,
        decoder_dims=config.DECODER_DIMS,
        clinical_output_dim=config.CLINICAL_OUTPUT_DIM,
        cluster_num=config.NUM_CLUSTERS,
        dropout=config.DROPOUT
    ).to(device)
    model.load_state_dict(state_dict)
    
    # Get numeric cluster predictions
    temp_df = clinical_df.copy()
    pred_numeric_labels = model.k_means_clustering(eval_loader, n_init=100, device=device)
    temp_df["mic_cluster"] = pred_numeric_labels
    
    # Map numeric predictions to subtype names
    temp_df, mapping = map_clusters_to_subtypes(temp_df, cluster_col_name='mic_cluster')
    
    if mapping is not None:
        # Calculate accuracy based on mapped subtype names
        pred_mapped_labels_numeric, _ = pd.factorize(temp_df["mapped_cluster"])
        accuracy = cluster_accuracy(true_labels_numeric, pred_mapped_labels_numeric)
        
        # Store the mapped predictions for this run
        col_name = f"mapped_cluster_{i+1}"
        all_mapped_predictions[col_name] = temp_df["mapped_cluster"]
        
        # If accuracy is high enough, mark this run as valid for majority vote
        if accuracy >= 0.75:
            valid_prediction_columns.append(col_name)
            print(f"Run {i+1}: Accuracy = {accuracy:.4f} -> VALID")
        else:
            print(f"Run {i+1}: Accuracy = {accuracy:.4f} -> Below threshold")
    else:
        print(f"Run {i+1}: Mapping failed, skipping.")

print(f"\nFound {len(valid_prediction_columns)} valid runs with accuracy >= 0.75.")

### 5.5. Map Final Labels and Save Results


In [ ]:
if not valid_prediction_columns:
    print("No valid runs found. Cannot perform majority vote.")
else:
    # Select only the predictions from valid runs
    valid_predictions_df = all_mapped_predictions[valid_prediction_columns]
    
    # Perform majority vote across the valid runs for each sample
    final_labels = valid_predictions_df.mode(axis=1)[0]
    
    # Add final labels to the main dataframe
    clinical_df["final_cluster"] = final_labels
    
    print("\nMajority vote complete.")
    print("Final subtype counts:\n", clinical_df["final_cluster"].value_counts())
    
    # --- Save Results ---
    export_df = clinical_df[[
        "id", "sex", "hba1c", "age_at_diagnosis", "bmi", 
        "HOMA_B", "HOMA_IR", "kmeans_cluster", "final_cluster"
    ]].copy()
    export_df.rename(columns={"final_cluster": "mic_cluster"}, inplace=True)
    
    # Save the final results to a CSV file
    results_save_path = config.PROCESSED_DATA_DIR / "final_clinical_data_with_labels.csv"
    export_df.to_csv(results_save_path, index=False)
    print(f"\nFinal results with stable labels saved to: {results_save_path}")

    # Save the accuracy list for the next notebook
    acc_save_path = config.OUTPUT_DIR / "model_accuracies.csv"
    pd.DataFrame({'accuracy': acc_list}).to_csv(acc_save_path, index=False)
    print(f"Model accuracies saved to: {acc_save_path}")